# USPS Texas Facilities - EDA and Deduplication

This notebook stages the USPS owned-facility inventory into separate dataframes:

- `raw_df`
- `clean_df`
- `eda_df`
- `collapsed_df`
- `near_matches_df`
- `missing_centroid_df`
- `final_df`

The original CSV is not mutated. The final export is written as `usps_texas_facilities_final.csv` in this folder.

In [1]:
from pathlib import Path
import csv
import re

import numpy as np
import pandas as pd
from IPython.display import Markdown, display

pd.set_option("display.max_columns", 120)
pd.set_option("display.max_colwidth", 140)
pd.set_option("display.max_rows", 250)

In [2]:
def first_existing_path(*relative_candidates):
    cwd = Path.cwd()
    search_roots = [cwd, cwd.parent, cwd.parent.parent, cwd.parent.parent.parent]
    for root in search_roots:
        for candidate in relative_candidates:
            path = (root / candidate).resolve()
            if path.exists():
                return path
    raise FileNotFoundError(f"Could not locate any of: {relative_candidates}")


USPS_CSV_PATH = first_existing_path(
    Path("USPS Post Offices in Texas.csv"),
    Path("Synthetic MySQL Tables/Facilities/USPS Post Offices in Texas.csv"),
)

TEXAS_ZIP_CENTROIDS_PATH = first_existing_path(
    Path("Texas_ZIP_Geo/texas_zip_centroids.csv"),
    Path("../../Texas_ZIP_Geo/texas_zip_centroids.csv"),
)

FINAL_EXPORT_PATH = USPS_CSV_PATH.with_name("usps_texas_facilities_final.csv")

print("USPS source:", USPS_CSV_PATH)
print("Texas ZIP centroids:", TEXAS_ZIP_CENTROIDS_PATH)
print("Final export path:", FINAL_EXPORT_PATH)

USPS source: Z:\Computer Science\GitHub Repositories\Personal Projects\Synthetic-Texas-Postal-Dataset\Synthetic MySQL Tables\Facilities\USPS Post Offices in Texas.csv
Texas ZIP centroids: Z:\Computer Science\GitHub Repositories\Personal Projects\Synthetic-Texas-Postal-Dataset\Texas_ZIP_Geo\texas_zip_centroids.csv
Final export path: Z:\Computer Science\GitHub Repositories\Personal Projects\Synthetic-Texas-Postal-Dataset\Synthetic MySQL Tables\Facilities\usps_texas_facilities_final.csv


## Helper Functions

In [3]:
CANONICAL_FACILITY_TYPES = [
    "Administrative Office",
    "Mail Processing",
    "Network Facilities",
    "Vehicle Maintenance",
    "Post Office",
]


def find_csv_header_row(path, first_column="District"):
    with open(path, "r", encoding="utf-8-sig", newline="") as handle:
        for row_number, row in enumerate(csv.reader(handle)):
            if row and row[0].strip() == first_column:
                return row_number
    raise ValueError(f"Could not find header row beginning with {first_column!r}")


def normalize_text(value):
    if pd.isna(value):
        return ""
    text = str(value).upper().strip()
    text = text.replace("&", " AND ")
    text = re.sub(r"[^A-Z0-9]+", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


def clean_object_columns(df):
    clean = df.copy()
    for column in clean.select_dtypes(include="object").columns:
        clean[column] = (
            clean[column]
            .astype("string")
            .str.replace(r"\s+", " ", regex=True)
            .str.strip()
            .replace({"": pd.NA, "nan": pd.NA, "None": pd.NA})
        )
    return clean


def unique_non_null(values):
    seen = []
    for value in values:
        if pd.isna(value):
            continue
        value = str(value).strip()
        if not value:
            continue
        if value not in seen:
            seen.append(value)
    return seen


def sorted_unique_non_null(values):
    return sorted(unique_non_null(values), key=lambda item: normalize_text(item))


def join_multivalue(values):
    values = unique_non_null(values)
    return "; ".join(values) if values else pd.NA


def choose_most_mentioned(values, prefer="text"):
    valid = []
    for value in values:
        if pd.isna(value):
            continue
        value = str(value).strip()
        if value:
            valid.append(value)
    if not valid:
        return pd.NA

    counts = pd.Series(valid).value_counts()
    top_count = counts.max()
    tied = counts[counts == top_count].index.tolist()

    if len(tied) == 1:
        return tied[0]

    if prefer == "numeric":
        parsed = pd.Series(tied).str.replace(",", "", regex=False).pipe(pd.to_numeric, errors="coerce")
        if parsed.notna().any():
            return tied[int(parsed.fillna(-np.inf).argmax())]

    if prefer == "date":
        parsed = pd.to_datetime(pd.Series(tied), errors="coerce")
        if parsed.notna().any():
            return tied[int(parsed.fillna(pd.Timestamp.min).argmax())]

    return sorted(tied, key=normalize_text)[-1]


def map_facility_type(value):
    if pd.isna(value):
        return pd.NA

    raw = str(value).strip()
    normalized = normalize_text(raw)

    exact_lookup = {normalize_text(item): item for item in CANONICAL_FACILITY_TYPES}
    if normalized in exact_lookup:
        return exact_lookup[normalized]

    if "OTHER CUSTOMER SERVICE" in normalized:
        return pd.NA
    if "ADMIN" in normalized:
        return "Administrative Office"
    if "MAIL" in normalized and "PROCESS" in normalized:
        return "Mail Processing"
    if "PROCESSING" in normalized and ("CENTER" in normalized or "FACILITY" in normalized):
        return "Mail Processing"
    if "NETWORK" in normalized or "LOGISTICS" in normalized:
        return "Network Facilities"
    if "VEHICLE" in normalized or "VMF" in normalized:
        return "Vehicle Maintenance"
    if "POST OFFICE" in normalized or normalized in {"STATION", "BRANCH"}:
        return "Post Office"

    return pd.NA


def extract_zip5(value):
    if pd.isna(value):
        return pd.NA
    match = re.search(r"(\d{5})", str(value))
    return match.group(1) if match else pd.NA


def extract_zip_suffix(value):
    if pd.isna(value):
        return pd.NA
    match = re.search(r"\d{5}[-\s]?(\d{1,4})", str(value))
    return match.group(1).zfill(4) if match else pd.NA

## Load and Clean Source CSV

In [4]:
header_row = find_csv_header_row(USPS_CSV_PATH)

raw_df = pd.read_csv(USPS_CSV_PATH, skiprows=header_row, dtype="string")
raw_df = raw_df.dropna(how="all").reset_index(drop=True)
raw_df["source_row_id"] = np.arange(len(raw_df))

print("raw rows:", len(raw_df))
display(raw_df.head())

raw rows: 688


,District,PO Name,Unit Name,Property Address,County,City,ST,ZIP Code,Ownership,FDB Facility Type (All),FDB Facility Subtype (All),Bldg Occu Date,Int Sq Ft,source_row_id
0,TEXAS 3,ABILENE,AUXILIARY VMF,810 N 4TH ST,TAYLOR,ABILENE,TX,79601-5924,Owned,Vehicle Maintenance,Vehicle Maintenance Facility (VMF),3/1/1987,"4,370",0
1,TEXAS 3,ABILENE,MAIN OFFICE,341 PINE ST,TAYLOR,ABILENE,TX,79601-9998,Owned,Post Office,Main Post Office,3/1/1936,"122,503",1
2,TEXAS 3,ABILENE,MAIN OFFICE,341 PINE ST,TAYLOR,ABILENE,TX,79601-9998,Owned,Post Office,Station,3/1/1936,"122,503",2
3,TEXAS 3,ABILENE,SOUTHERN HILLS STA,2501 BUFFALO GAP RD,TAYLOR,ABILENE,TX,79605-6185,Owned,Post Office,Station,12/1/1984,"17,130",3
4,TEXAS 1,ADDISON,MAIN OFFICE,4900 AIRPORT PKWY,DALLAS,ADDISON,TX,75001-9998,Owned,Post Office,Main Post Office,8/1/1998,"18,564",4


In [5]:
clean_df = clean_object_columns(raw_df)

clean_df["zip5"] = clean_df["ZIP Code"].map(extract_zip5)
clean_df["zip_suffix"] = clean_df["ZIP Code"].map(extract_zip_suffix)
clean_df["normalized_po_name"] = clean_df["PO Name"].map(normalize_text)
clean_df["normalized_unit_name"] = clean_df["Unit Name"].map(normalize_text)
clean_df["normalized_property_address"] = clean_df["Property Address"].map(normalize_text)
clean_df["normalized_county"] = clean_df["County"].map(normalize_text)
clean_df["normalized_city"] = clean_df["City"].map(normalize_text)
clean_df["normalized_facility_type"] = clean_df["FDB Facility Type (All)"].map(normalize_text)
clean_df["mapped_facility_type"] = clean_df["FDB Facility Type (All)"].map(map_facility_type)

facility_identity_columns = [
    "normalized_po_name",
    "normalized_property_address",
    "normalized_county",
    "normalized_city",
]

clean_df["facility_base_key"] = clean_df[facility_identity_columns].agg("|".join, axis=1)
clean_df["facility_zip_key"] = clean_df[facility_identity_columns + ["zip5"]].fillna("").agg("|".join, axis=1)

print("clean rows:", len(clean_df))
display(clean_df.head())

clean rows: 688


,District,PO Name,Unit Name,Property Address,County,City,ST,ZIP Code,Ownership,FDB Facility Type (All),FDB Facility Subtype (All),Bldg Occu Date,Int Sq Ft,source_row_id,zip5,zip_suffix,normalized_po_name,normalized_unit_name,normalized_property_address,normalized_county,normalized_city,normalized_facility_type,mapped_facility_type,facility_base_key,facility_zip_key
0,TEXAS 3,ABILENE,AUXILIARY VMF,810 N 4TH ST,TAYLOR,ABILENE,TX,79601-5924,Owned,Vehicle Maintenance,Vehicle Maintenance Facility (VMF),3/1/1987,"4,370",0,79601,5924,ABILENE,AUXILIARY VMF,810 N 4TH ST,TAYLOR,ABILENE,VEHICLE MAINTENANCE,Vehicle Maintenance,ABILENE|810 N 4TH ST|TAYLOR|ABILENE,ABILENE|810 N 4TH ST|TAYLOR|ABILENE|79601
1,TEXAS 3,ABILENE,MAIN OFFICE,341 PINE ST,TAYLOR,ABILENE,TX,79601-9998,Owned,Post Office,Main Post Office,3/1/1936,"122,503",1,79601,9998,ABILENE,MAIN OFFICE,341 PINE ST,TAYLOR,ABILENE,POST OFFICE,Post Office,ABILENE|341 PINE ST|TAYLOR|ABILENE,ABILENE|341 PINE ST|TAYLOR|ABILENE|79601
2,TEXAS 3,ABILENE,MAIN OFFICE,341 PINE ST,TAYLOR,ABILENE,TX,79601-9998,Owned,Post Office,Station,3/1/1936,"122,503",2,79601,9998,ABILENE,MAIN OFFICE,341 PINE ST,TAYLOR,ABILENE,POST OFFICE,Post Office,ABILENE|341 PINE ST|TAYLOR|ABILENE,ABILENE|341 PINE ST|TAYLOR|ABILENE|79601
3,TEXAS 3,ABILENE,SOUTHERN HILLS STA,2501 BUFFALO GAP RD,TAYLOR,ABILENE,TX,79605-6185,Owned,Post Office,Station,12/1/1984,"17,130",3,79605,6185,ABILENE,SOUTHERN HILLS STA,2501 BUFFALO GAP RD,TAYLOR,ABILENE,POST OFFICE,Post Office,ABILENE|2501 BUFFALO GAP RD|TAYLOR|ABILENE,ABILENE|2501 BUFFALO GAP RD|TAYLOR|ABILENE|79605
4,TEXAS 1,ADDISON,MAIN OFFICE,4900 AIRPORT PKWY,DALLAS,ADDISON,TX,75001-9998,Owned,Post Office,Main Post Office,8/1/1998,"18,564",4,75001,9998,ADDISON,MAIN OFFICE,4900 AIRPORT PKWY,DALLAS,ADDISON,POST OFFICE,Post Office,ADDISON|4900 AIRPORT PKWY|DALLAS|ADDISON,ADDISON|4900 AIRPORT PKWY|DALLAS|ADDISON|75001


## Simple EDA

In [6]:
eda_df = clean_df.copy()

print("eda rows:", len(eda_df))
display(
    pd.DataFrame(
        {
            "metric": [
                "rows",
                "columns",
                "unique facility base keys",
                "unique facility zip keys",
                "unique zip5 values",
            ],
            "value": [
                len(eda_df),
                len(eda_df.columns),
                eda_df["facility_base_key"].nunique(dropna=True),
                eda_df["facility_zip_key"].nunique(dropna=True),
                eda_df["zip5"].nunique(dropna=True),
            ],
        }
    )
)

display(eda_df.dtypes.rename("dtype").reset_index().rename(columns={"index": "column"}))

eda rows: 688


,metric,value
0,rows,688
1,columns,25
2,unique facility base keys,586
3,unique facility zip keys,587
4,unique zip5 values,569


,column,dtype
0,District,string
1,PO Name,string
2,Unit Name,string
3,Property Address,string
4,County,string
5,City,string
6,ST,string
7,ZIP Code,string
8,Ownership,string
9,FDB Facility Type (All),string


In [7]:
facility_type_values = (
    eda_df["FDB Facility Type (All)"]
    .value_counts(dropna=False)
    .rename_axis("FDB Facility Type (All)")
    .reset_index(name="record_count")
)

facility_subtype_values = (
    eda_df["FDB Facility Subtype (All)"]
    .value_counts(dropna=False)
    .rename_axis("FDB Facility Subtype (All)")
    .reset_index(name="record_count")
)

display(Markdown("### FDB Facility Type (All) values"))
display(facility_type_values)

display(Markdown("### FDB Facility Subtype (All) values"))
display(facility_subtype_values)

### FDB Facility Type (All) values

,FDB Facility Type (All),record_count
0,Post Office,594
1,Vehicle Maintenance,24
2,Administrative Office,20
3,Network Facilities,19
4,Mail Processing,16
5,<NA>,13
6,Other Customer Service,2


### FDB Facility Subtype (All) values

,FDB Facility Subtype (All),record_count
0,Main Post Office,156
1,Station,148
2,Administrative Post Office (APO),119
3,Remotely Managed Post Office (RMPO),94
4,Vehicle Maintenance Facility (VMF),24
5,Finance Station,22
6,Branch,18
7,Logistics (LOG),16
8,Processing and Distribution Center/Facility (PDC/PDF),13
9,<NA>,13


In [8]:
type_subtype_counts = (
    eda_df.groupby(["FDB Facility Type (All)", "FDB Facility Subtype (All)"], dropna=False)
    .size()
    .reset_index(name="record_count")
    .sort_values(["FDB Facility Type (All)", "FDB Facility Subtype (All)"], na_position="last")
)

tree_lines = []
for facility_type, type_group in type_subtype_counts.groupby("FDB Facility Type (All)", dropna=False):
    type_label = "<NULL>" if pd.isna(facility_type) else str(facility_type)
    type_total = int(type_group["record_count"].sum())
    tree_lines.append(f"{type_label} ({type_total})")
    for _, row in type_group.iterrows():
        subtype = row["FDB Facility Subtype (All)"]
        subtype_label = "<NULL>" if pd.isna(subtype) else str(subtype)
        tree_lines.append(f"  |-- {subtype_label} ({int(row['record_count'])})")

display(Markdown("### Facility Type/Subtype hierarchy"))
print("\n".join(tree_lines))
display(type_subtype_counts)

### Facility Type/Subtype hierarchy

Administrative Office (20)
  |-- Area Administrative Office (AVP) (1)
  |-- Consumer Affairs Headquarters (1)
  |-- Consumer Affairs Office (2)
  |-- District Administrative Office (DM) (3)
  |-- Division Administrative Office (Logistics) (1)
  |-- Division Administrative Office (Processing) (1)
  |-- Facilities Service Office (FSO) (1)
  |-- Headquarters Related (1)
  |-- Office of the Inspector General Office (OIG) (5)
  |-- Postal Inspection Service Field Division (USPIS) (4)
Mail Processing (16)
  |-- Local Processing Center (LPC_NO) (1)
  |-- Mail Processing Annex (ANX) (2)
  |-- Processing and Distribution Center/Facility (PDC/PDF) (13)
Network Facilities (19)
  |-- Hub Facility (HUB) (1)
  |-- Logistics (LOG) (16)
  |-- Network Distribution Center (NDC/ASF) (1)
  |-- Regional Processing and Distribution Center (RPDC) (1)
Other Customer Service (2)
  |-- Computerized Forwarding System (CFS) (2)
Post Office (594)
  |-- Administrative Post Office (APO) (119)
  |-- Branch (18)
  |--

,FDB Facility Type (All),FDB Facility Subtype (All),record_count
0,Administrative Office,Area Administrative Office (AVP),1
1,Administrative Office,Consumer Affairs Headquarters,1
2,Administrative Office,Consumer Affairs Office,2
3,Administrative Office,District Administrative Office (DM),3
4,Administrative Office,Division Administrative Office (Logistics),1
5,Administrative Office,Division Administrative Office (Processing),1
6,Administrative Office,Facilities Service Office (FSO),1
7,Administrative Office,Headquarters Related,1
8,Administrative Office,Office of the Inspector General Office (OIG),5
9,Administrative Office,Postal Inspection Service Field Division (USPIS),4


In [9]:
ownership_summary = (
    eda_df["Ownership"]
    .fillna("<NULL>")
    .value_counts(dropna=False)
    .rename_axis("Ownership")
    .reset_index(name="record_count")
)

owned_vs_other_summary = pd.DataFrame(
    {
        "ownership_group": ["Owned", "Other or NULL"],
        "record_count": [
            int(eda_df["Ownership"].eq("Owned").sum()),
            int((~eda_df["Ownership"].eq("Owned") | eda_df["Ownership"].isna()).sum()),
        ],
    }
)

display(Markdown("### Ownership counts"))
display(ownership_summary)
display(owned_vs_other_summary)

### Ownership counts

,Ownership,record_count
0,Owned,686
1,<NULL>,2


,ownership_group,record_count
0,Owned,686
1,Other or NULL,2


In [10]:
unique_zip5_count = eda_df["zip5"].nunique(dropna=True)
zip5_summary = (
    eda_df.groupby("zip5", dropna=False)
    .agg(
        record_count=("source_row_id", "count"),
        unique_facility_base_count=("facility_base_key", "nunique"),
        zip_suffixes=("zip_suffix", lambda values: sorted_unique_non_null(values)),
    )
    .reset_index()
    .sort_values(["zip5"], na_position="last")
)
zip5_summary["zip_suffix_count"] = zip5_summary["zip_suffixes"].map(len)

same_facility_different_suffixes = (
    eda_df.groupby(["zip5", "facility_base_key"], dropna=False)
    .agg(
        record_count=("source_row_id", "count"),
        po_names=("PO Name", sorted_unique_non_null),
        property_addresses=("Property Address", sorted_unique_non_null),
        cities=("City", sorted_unique_non_null),
        zip_codes_seen=("ZIP Code", sorted_unique_non_null),
        zip_suffixes=("zip_suffix", sorted_unique_non_null),
    )
    .reset_index()
)
same_facility_different_suffixes["zip_suffix_count"] = same_facility_different_suffixes["zip_suffixes"].map(len)
same_facility_different_suffixes = same_facility_different_suffixes[
    same_facility_different_suffixes["zip_suffix_count"] > 1
].sort_values(["zip5", "facility_base_key"])

print("unique ZIP prefixes:", unique_zip5_count)
display(Markdown("### ZIP prefix summary"))
display(zip5_summary)

display(Markdown("### Same facility records with multiple ZIP suffixes"))
display(same_facility_different_suffixes)

unique ZIP prefixes: 569


### ZIP prefix summary

,zip5,record_count,unique_facility_base_count,zip_suffixes,zip_suffix_count
0,75001,1,1,[9998],1
1,75006,1,1,[9998],1
2,75007,1,1,[9998],1
3,75013,1,1,[8043],1
4,75019,1,1,[9998],1
...,...,...,...,...,...
565,79910,6,2,"[9716, 9721, 9998]",3
566,79912,1,1,[9998],1
567,79925,1,1,[9998],1
568,79936,1,1,[9998],1


### Same facility records with multiple ZIP suffixes

,zip5,facility_base_key,record_count,po_names,property_addresses,cities,zip_codes_seen,zip_suffixes,zip_suffix_count
36,75099,NORTH TEXAS|951 W BETHEL RD|DALLAS|COPPELL,9,[NORTH TEXAS],[951 W BETHEL RD],[COPPELL],"[75099-5000, 75099-9301]","[5000, 9301]",2
74,75260,DALLAS|401 TOM LANDRY HWY|DALLAS|DALLAS,8,[DALLAS],[401 TOM LANDRY HWY],[DALLAS],"[75260-9731, 75260-9998]","[9731, 9998]",2
93,75501,TEXARKANA|2211 N ROBISON RD|BOWIE|TEXARKANA,4,[TEXARKANA],[2211 N ROBISON RD],[TEXARKANA],"[75501-9995, 75501-9998]","[9995, 9998]",2
112,75702,TYLER|2100 W MARTIN LUTHER KING JR BLVD|SMITH|TYLER,2,[TYLER],[2100 W MARTIN LUTHER KING JR BLVD],[TYLER],"[75702-9351, 75702-9998]","[9351, 9998]",2
158,76105,FORT WORTH|4650 E ROSEDALE ST|TARRANT|FORT WORTH,2,[FORT WORTH],[4650 E ROSEDALE ST],[FORT WORTH],"[76105-1800, 76105-9998]","[1800, 9998]",2
172,76161,FORT WORTH|4600 MARK IV PKWY|TARRANT|FORT WORTH,5,[FORT WORTH],[4600 MARK IV PKWY],[FORT WORTH],"[76161-9997, 76161-9998]","[9997, 9998]",2
232,76702,WACO|430 W STATE HIGHWAY 6|MCLENNAN|WACO,3,[WACO],[430 W STATE HIGHWAY 6],[WACO],"[76702-0009, 76702-9304]","[0009, 9304]",2
248,76902,SAN ANGELO|1 N ABE ST|TOM GREEN|SAN ANGELO,2,[SAN ANGELO],[1 N ABE ST],[SAN ANGELO],"[76902-9700, 76902-9998]","[9700, 9998]",2
261,77017,HOUSTON|5302 GALVESTON RD|HARRIS|HOUSTON,2,[HOUSTON],[5302 GALVESTON RD],[HOUSTON],"[77017-9995, 77017-9998]","[9995, 9998]",2
290,77084,HOUSTON|16015 CAIRNWAY DR|HARRIS|HOUSTON,2,[HOUSTON],[16015 CAIRNWAY DR],[HOUSTON],"[77084-9995, 77084-9998]","[9995, 9998]",2


## Remove Facilities Already Manually Entered

The target list below intentionally includes aliases and optional city/ZIP/address constraints. Review `manual_exclusion_candidates` before relying on the final export.

In [11]:
MANUAL_FACILITY_TARGETS = [
    {
        "label": "Sam Houston Post Office",
        "aliases": ["SAM HOUSTON", "HOUSTON MAIN OFFICE", "MAIN OFFICE"],
        "city": "HOUSTON",
        "zip5": "77002",
    },
    {
        "label": "Eastwood Post Office",
        "aliases": ["EASTWOOD"],
        "city": "HOUSTON",
    },
    {
        "label": "T W House Post Office",
        "aliases": ["T W HOUSE", "TW HOUSE"],
        "city": "HOUSTON",
    },
    {
        "label": "USPS North Houston TX Distribution Center",
        "aliases": ["N HOUSTON TX RPDC", "NORTH HOUSTON TX RPDC", "NORTH HOUSTON DISTRIBUTION CENTER"],
        "city": "NORTH HOUSTON",
        "zip5": "77315",
    },
    {
        "label": "Roy Royall Post Office",
        "aliases": ["ROY ROYALL"],
        "city": "HOUSTON",
    },
    {
        "label": "Astrodome Post Office",
        "aliases": ["ASTRODOME"],
        "city": "HOUSTON",
    },
]


def manual_target_match(row, targets=MANUAL_FACILITY_TARGETS):
    searchable_text = normalize_text(
        " | ".join(
            str(row.get(column, ""))
            for column in ["PO Name", "Unit Name", "Property Address", "City", "ZIP Code"]
            if not pd.isna(row.get(column, pd.NA))
        )
    )

    matched_labels = []
    for target in targets:
        target_city = target.get("city")
        if target_city and normalize_text(row.get("City", "")) != normalize_text(target_city):
            continue

        target_zip5 = target.get("zip5")
        if target_zip5 and row.get("zip5", pd.NA) != target_zip5:
            continue

        target_address_contains = target.get("address_contains")
        if target_address_contains and normalize_text(target_address_contains) not in normalize_text(row.get("Property Address", "")):
            continue

        for alias in target["aliases"]:
            if normalize_text(alias) in searchable_text:
                matched_labels.append(target["label"])
                break

    return matched_labels


clean_df["manual_exclusion_matches"] = clean_df.apply(manual_target_match, axis=1)
clean_df["is_manual_exclusion"] = clean_df["manual_exclusion_matches"].map(bool)

manual_exclusion_candidates = clean_df.loc[
    clean_df["is_manual_exclusion"],
    [
        "source_row_id",
        "PO Name",
        "Unit Name",
        "Property Address",
        "County",
        "City",
        "ZIP Code",
        "FDB Facility Type (All)",
        "FDB Facility Subtype (All)",
        "manual_exclusion_matches",
    ],
].copy()

display(manual_exclusion_candidates)

dedupe_input_df = clean_df.loc[~clean_df["is_manual_exclusion"]].copy()

print("clean rows:", len(clean_df))
print("manual exclusion rows:", len(manual_exclusion_candidates))
print("dedupe input rows:", len(dedupe_input_df))

,source_row_id,PO Name,Unit Name,Property Address,County,City,ZIP Code,FDB Facility Type (All),FDB Facility Subtype (All),manual_exclusion_matches
315,315,HOUSTON,ASTRODOME STA EXP,8205 BRAESMAIN DR,HARRIS,HOUSTON,77025-9998,Post Office,Station,[Astrodome Post Office]
325,325,HOUSTON,EASTWOOD STATION,5415 LAWNDALE ST,HARRIS,HOUSTON,77023-9998,Post Office,Station,[Eastwood Post Office]
337,337,HOUSTON,MAIN OFFICE,1500 HADLEY ST,HOUSTON,HOUSTON,77002-9998,Post Office,Carrier Annex (ANX),[Sam Houston Post Office]
341,341,HOUSTON,N HOUSTON TX RPDC,4600 ALDINE BENDER RD,HARRIS,NORTH HOUSTON,77315-9900,Post Office,Main Post Office,[USPS North Houston TX Distribution Center]
342,342,HOUSTON,N HOUSTON TX RPDC,4600 ALDINE BENDER RD,HARRIS,NORTH HOUSTON,77315-9900,Administrative Office,Postal Inspection Service Field Division (USPIS),[USPS North Houston TX Distribution Center]
343,343,HOUSTON,N HOUSTON TX RPDC,4600 ALDINE BENDER RD,HARRIS,NORTH HOUSTON,77315-9900,Network Facilities,Regional Processing and Distribution Center (RPDC),[USPS North Houston TX Distribution Center]
344,344,HOUSTON,N HOUSTON TX RPDC,4600 ALDINE BENDER RD,HARRIS,NORTH HOUSTON,77315-9900,Administrative Office,District Administrative Office (DM),[USPS North Houston TX Distribution Center]
345,345,HOUSTON,N HOUSTON TX RPDC,4600 ALDINE BENDER RD,HARRIS,NORTH HOUSTON,77315-9900,Administrative Office,Office of the Inspector General Office (OIG),[USPS North Houston TX Distribution Center]
346,346,HOUSTON,N HOUSTON TX RPDC,4600 ALDINE BENDER RD,HARRIS,NORTH HOUSTON,77315-9900,Network Facilities,Logistics (LOG),[USPS North Houston TX Distribution Center]
347,347,HOUSTON,N HOUSTON TX RPDC,4600 ALDINE BENDER RD,HARRIS,NORTH HOUSTON,77315-9900,Mail Processing,Local Processing Center (LPC_NO),[USPS North Houston TX Distribution Center]


clean rows: 688
manual exclusion rows: 12
dedupe input rows: 676


## Collapse to One Record per Unique Facility

In [12]:
def collapse_facility_group(group):
    review_reasons = []

    zip5_values = unique_non_null(group["zip5"])
    zip_suffix_values = unique_non_null(group["zip_suffix"])
    unit_name_values = unique_non_null(group["Unit Name"])
    sqft_values = unique_non_null(group["Int Sq Ft"])
    occupancy_date_values = unique_non_null(group["Bldg Occu Date"])

    if len(zip5_values) > 1:
        review_reasons.append("MULTIPLE_ZIP5_FOR_SAME_NAME_ADDRESS")
    if len(zip_suffix_values) > 1:
        review_reasons.append("MULTIPLE_ZIP_SUFFIXES")
    if len(unit_name_values) > 1:
        review_reasons.append("MULTIPLE_UNIT_NAMES")
    if len(sqft_values) > 1:
        review_reasons.append("CONFLICTING_SQFT")
    if len(occupancy_date_values) > 1:
        review_reasons.append("CONFLICTING_OCCUPANCY_DATE")

    mapped_types = sorted_unique_non_null(group["mapped_facility_type"])
    raw_types = sorted_unique_non_null(group["FDB Facility Type (All)"])
    raw_subtypes = sorted_unique_non_null(group["FDB Facility Subtype (All)"])

    if not mapped_types:
        review_reasons.append("UNKNOWN_FACILITY_TYPE")

    return pd.Series(
        {
            "District": choose_most_mentioned(group["District"]),
            "PO Name": choose_most_mentioned(group["PO Name"]),
            "Property Address": choose_most_mentioned(group["Property Address"]),
            "County": choose_most_mentioned(group["County"]),
            "City": choose_most_mentioned(group["City"]),
            "ST": choose_most_mentioned(group["ST"]),
            "ZIP Code": choose_most_mentioned(group["ZIP Code"]),
            "zip5": choose_most_mentioned(group["zip5"]),
            "zip_suffix": choose_most_mentioned(group["zip_suffix"]),
            "Ownership": choose_most_mentioned(group["Ownership"]),
            "facility_types": mapped_types,
            "facility_subtypes": raw_subtypes,
            "unit_names_seen": sorted_unique_non_null(group["Unit Name"]),
            "Int Sq Ft": choose_most_mentioned(group["Int Sq Ft"], prefer="numeric"),
            "Bldg Occu Date": choose_most_mentioned(group["Bldg Occu Date"], prefer="date"),
            "source_row_count": len(group),
            "source_row_ids": sorted_unique_non_null(group["source_row_id"].astype(str)),
            "original_zip_codes": sorted_unique_non_null(group["ZIP Code"]),
            "original_facility_types": raw_types,
            "original_facility_subtypes": raw_subtypes,
            "review_reasons": review_reasons,
        }
    )


collapsed_df = (
    dedupe_input_df.groupby("facility_base_key", dropna=False)
    .apply(collapse_facility_group)
    .reset_index(drop=True)
)

collapsed_df["has_known_facility_type"] = collapsed_df["facility_types"].map(bool)

print("collapsed rows:", len(collapsed_df))
display(collapsed_df.head())

collapsed rows: 580


,District,PO Name,Property Address,County,City,ST,ZIP Code,zip5,zip_suffix,Ownership,facility_types,facility_subtypes,unit_names_seen,Int Sq Ft,Bldg Occu Date,source_row_count,source_row_ids,original_zip_codes,original_facility_types,original_facility_subtypes,review_reasons,has_known_facility_type
0,TEXAS 3,ABILENE,2501 BUFFALO GAP RD,TAYLOR,ABILENE,TX,79605-6185,79605,6185,Owned,[Post Office],[Station],[SOUTHERN HILLS STA],"17,130",12/1/1984,1,[3],[79605-6185],[Post Office],[Station],[],True
1,TEXAS 3,ABILENE,341 PINE ST,TAYLOR,ABILENE,TX,79601-9998,79601,9998,Owned,[Post Office],"[Main Post Office, Station]",[MAIN OFFICE],"122,503",3/1/1936,2,"[1, 2]",[79601-9998],[Post Office],"[Main Post Office, Station]",[],True
2,TEXAS 3,ABILENE,810 N 4TH ST,TAYLOR,ABILENE,TX,79601-5924,79601,5924,Owned,[Vehicle Maintenance],[Vehicle Maintenance Facility (VMF)],[AUXILIARY VMF],"4,370",3/1/1987,1,[0],[79601-5924],[Vehicle Maintenance],[Vehicle Maintenance Facility (VMF)],[],True
3,TEXAS 1,ADDISON,4900 AIRPORT PKWY,DALLAS,ADDISON,TX,75001-9998,75001,9998,Owned,[Post Office],[Main Post Office],[MAIN OFFICE],"18,564",8/1/1998,1,[4],[75001-9998],[Post Office],[Main Post Office],[],True
4,TEXAS 2,ALAMO,423 LOS ALAMOS DR,HIDALGO,ALAMO,TX,78516-9998,78516,9998,Owned,[Post Office],[Main Post Office],[MAIN OFFICE],"5,049",NaN,1,[5],[78516-9998],[Post Office],[Main Post Office],[],True


## Centroid Coverage and Final Export

In [13]:
centroids_df = pd.read_csv(TEXAS_ZIP_CENTROIDS_PATH, dtype={"zip": "string"})
centroids_df["zip5"] = centroids_df["zip"].map(extract_zip5)
centroid_zip5_values = set(centroids_df["zip5"].dropna())

candidate_final_df = collapsed_df.loc[collapsed_df["has_known_facility_type"]].copy()
candidate_final_df["has_centroid_zip"] = candidate_final_df["zip5"].isin(centroid_zip5_values)

missing_centroid_df = candidate_final_df.loc[~candidate_final_df["has_centroid_zip"]].copy()
missing_centroid_df["review_reasons"] = missing_centroid_df["review_reasons"].map(
    lambda reasons: sorted(set(list(reasons) + ["MISSING_CENTROID_ZIP"]))
)

final_df = candidate_final_df.loc[candidate_final_df["has_centroid_zip"]].copy()

near_matches_df = pd.concat(
    [
        collapsed_df.loc[
            collapsed_df["review_reasons"].map(bool)
            | ~collapsed_df["has_known_facility_type"]
        ].copy(),
        missing_centroid_df.copy(),
    ],
    ignore_index=True,
)

near_matches_df["review_reasons"] = near_matches_df["review_reasons"].map(
    lambda reasons: sorted(set(reasons)) if isinstance(reasons, list) else []
)
near_matches_df["near_match_key"] = near_matches_df[["PO Name", "Property Address", "County", "City", "zip5"]].fillna("").agg("|".join, axis=1)

def union_review_reasons(reason_series):
    reasons = []
    for value in reason_series:
        if isinstance(value, list):
            reasons.extend(value)
    return sorted(set(reasons))

review_reasons_by_key = near_matches_df.groupby("near_match_key")["review_reasons"].apply(union_review_reasons)
near_matches_df = near_matches_df.drop_duplicates(subset=["near_match_key"], keep="first")
near_matches_df["review_reasons"] = near_matches_df["near_match_key"].map(review_reasons_by_key)
near_matches_df = near_matches_df.drop(columns=["near_match_key"])

print("collapsed rows:", len(collapsed_df))
print("near matches:", len(near_matches_df))
print("missing centroid rows:", len(missing_centroid_df))
print("final rows:", len(final_df))

display(Markdown("### ZIP centroid coverage"))
display(
    pd.DataFrame(
        {
            "metric": ["candidate final rows", "covered by texas_zip_centroids", "missing from texas_zip_centroids"],
            "value": [len(candidate_final_df), len(final_df), len(missing_centroid_df)],
        }
    )
)

collapsed rows: 580
near matches: 59
missing centroid rows: 40
final rows: 533


### ZIP centroid coverage

,metric,value
0,candidate final rows,573
1,covered by texas_zip_centroids,533
2,missing from texas_zip_centroids,40


In [14]:
list_columns = [
    "facility_types",
    "facility_subtypes",
    "unit_names_seen",
    "source_row_ids",
    "original_zip_codes",
    "original_facility_types",
    "original_facility_subtypes",
    "review_reasons",
]

final_export_df = final_df.copy()
for column in list_columns:
    if column in final_export_df.columns:
        final_export_df[column] = final_export_df[column].map(lambda values: "; ".join(values) if isinstance(values, list) else values)

final_export_df = final_export_df.drop(columns=["has_known_facility_type", "has_centroid_zip"], errors="ignore")
final_export_df.to_csv(FINAL_EXPORT_PATH, index=False)

print("final rows:", len(final_df))
print("exported:", FINAL_EXPORT_PATH)
display(final_export_df.head())

final rows: 533
exported: Z:\Computer Science\GitHub Repositories\Personal Projects\Synthetic-Texas-Postal-Dataset\Synthetic MySQL Tables\Facilities\usps_texas_facilities_final.csv


,District,PO Name,Property Address,County,City,ST,ZIP Code,zip5,zip_suffix,Ownership,facility_types,facility_subtypes,unit_names_seen,Int Sq Ft,Bldg Occu Date,source_row_count,source_row_ids,original_zip_codes,original_facility_types,original_facility_subtypes,review_reasons
0,TEXAS 3,ABILENE,2501 BUFFALO GAP RD,TAYLOR,ABILENE,TX,79605-6185,79605,6185,Owned,Post Office,Station,SOUTHERN HILLS STA,"17,130",12/1/1984,1,3,79605-6185,Post Office,Station,
1,TEXAS 3,ABILENE,341 PINE ST,TAYLOR,ABILENE,TX,79601-9998,79601,9998,Owned,Post Office,Main Post Office; Station,MAIN OFFICE,"122,503",3/1/1936,2,1; 2,79601-9998,Post Office,Main Post Office; Station,
2,TEXAS 3,ABILENE,810 N 4TH ST,TAYLOR,ABILENE,TX,79601-5924,79601,5924,Owned,Vehicle Maintenance,Vehicle Maintenance Facility (VMF),AUXILIARY VMF,"4,370",3/1/1987,1,0,79601-5924,Vehicle Maintenance,Vehicle Maintenance Facility (VMF),
3,TEXAS 1,ADDISON,4900 AIRPORT PKWY,DALLAS,ADDISON,TX,75001-9998,75001,9998,Owned,Post Office,Main Post Office,MAIN OFFICE,"18,564",8/1/1998,1,4,75001-9998,Post Office,Main Post Office,
4,TEXAS 2,ALAMO,423 LOS ALAMOS DR,HIDALGO,ALAMO,TX,78516-9998,78516,9998,Owned,Post Office,Main Post Office,MAIN OFFICE,"5,049",NaN,1,5,78516-9998,Post Office,Main Post Office,


## Review Dataframes

In [15]:
display(Markdown("### manual_exclusion_candidates.head()"))
display(manual_exclusion_candidates.head(25))

display(Markdown("### near_matches_df.head()"))
display(near_matches_df.head(25))

display(Markdown("### missing_centroid_df.head()"))
display(missing_centroid_df.head(25))

display(Markdown("### final_df.head()"))
display(final_df.head(25))

### manual_exclusion_candidates.head()

,source_row_id,PO Name,Unit Name,Property Address,County,City,ZIP Code,FDB Facility Type (All),FDB Facility Subtype (All),manual_exclusion_matches
315,315,HOUSTON,ASTRODOME STA EXP,8205 BRAESMAIN DR,HARRIS,HOUSTON,77025-9998,Post Office,Station,[Astrodome Post Office]
325,325,HOUSTON,EASTWOOD STATION,5415 LAWNDALE ST,HARRIS,HOUSTON,77023-9998,Post Office,Station,[Eastwood Post Office]
337,337,HOUSTON,MAIN OFFICE,1500 HADLEY ST,HOUSTON,HOUSTON,77002-9998,Post Office,Carrier Annex (ANX),[Sam Houston Post Office]
341,341,HOUSTON,N HOUSTON TX RPDC,4600 ALDINE BENDER RD,HARRIS,NORTH HOUSTON,77315-9900,Post Office,Main Post Office,[USPS North Houston TX Distribution Center]
342,342,HOUSTON,N HOUSTON TX RPDC,4600 ALDINE BENDER RD,HARRIS,NORTH HOUSTON,77315-9900,Administrative Office,Postal Inspection Service Field Division (USPIS),[USPS North Houston TX Distribution Center]
343,343,HOUSTON,N HOUSTON TX RPDC,4600 ALDINE BENDER RD,HARRIS,NORTH HOUSTON,77315-9900,Network Facilities,Regional Processing and Distribution Center (RPDC),[USPS North Houston TX Distribution Center]
344,344,HOUSTON,N HOUSTON TX RPDC,4600 ALDINE BENDER RD,HARRIS,NORTH HOUSTON,77315-9900,Administrative Office,District Administrative Office (DM),[USPS North Houston TX Distribution Center]
345,345,HOUSTON,N HOUSTON TX RPDC,4600 ALDINE BENDER RD,HARRIS,NORTH HOUSTON,77315-9900,Administrative Office,Office of the Inspector General Office (OIG),[USPS North Houston TX Distribution Center]
346,346,HOUSTON,N HOUSTON TX RPDC,4600 ALDINE BENDER RD,HARRIS,NORTH HOUSTON,77315-9900,Network Facilities,Logistics (LOG),[USPS North Houston TX Distribution Center]
347,347,HOUSTON,N HOUSTON TX RPDC,4600 ALDINE BENDER RD,HARRIS,NORTH HOUSTON,77315-9900,Mail Processing,Local Processing Center (LPC_NO),[USPS North Houston TX Distribution Center]


### near_matches_df.head()

,District,PO Name,Property Address,County,City,ST,ZIP Code,zip5,zip_suffix,Ownership,facility_types,facility_subtypes,unit_names_seen,Int Sq Ft,Bldg Occu Date,source_row_count,source_row_ids,original_zip_codes,original_facility_types,original_facility_subtypes,review_reasons,has_known_facility_type,has_centroid_zip
0,TEXAS 3,AMARILLO,2301 ROSS ST,POTTER,AMARILLO,TX,79120-9998,79120,9998,Owned,"[Mail Processing, Network Facilities, Post Office, Vehicle Maintenance]","[Logistics (LOG), Main Post Office, Processing and Distribution Center/Facility (PDC/PDF), Vehicle Maintenance Facility (VMF)]","[PDC, VMF]","220,270",3/1/1977,4,"[12, 13, 14, 15]","[79120-9341, 79120-9998]","[Mail Processing, Network Facilities, Post Office, Vehicle Maintenance]","[Logistics (LOG), Main Post Office, Processing and Distribution Center/Facility (PDC/PDF), Vehicle Maintenance Facility (VMF)]","[CONFLICTING_SQFT, MISSING_CENTROID_ZIP, MULTIPLE_UNIT_NAMES, MULTIPLE_ZIP_SUFFIXES]",True,NaN
1,TEXAS 1,ARLINGTON,3903 MELEAR DR,TARRANT,ARLINGTON,TX,76015-9998,76015,9998,Owned,[Post Office],[Station],"[MELEAR STATION, MELEAR STORAGE]","28,666",2/1/1988,2,"[21, 22]",[76015-9998],[Post Office],[Station],"[CONFLICTING_OCCUPANCY_DATE, CONFLICTING_SQFT, MULTIPLE_UNIT_NAMES]",True,NaN
2,TEXAS 3,AUSTIN,900 BLACKSON AVE,TRAVIS,AUSTIN,TX,78752-9998,78752,9998,Owned,"[Post Office, Vehicle Maintenance]","[Station, Vehicle Maintenance Facility (VMF)]","[NORTHEAST STATION, NORTHEAST VMF]","14,875",10/1/1974,2,"[39, 40]",[78752-9998],"[Post Office, Vehicle Maintenance]","[Station, Vehicle Maintenance Facility (VMF)]","[CONFLICTING_SQFT, MULTIPLE_UNIT_NAMES]",True,NaN
3,TEXAS 2,BEAUMONT,5815 WALDEN RD,JEFFERSON,BEAUMONT,TX,77707-9998,77707,9998,Owned,"[Mail Processing, Network Facilities, Post Office, Vehicle Maintenance]","[Logistics (LOG), Main Post Office, Processing and Distribution Center/Facility (PDC/PDF), Service Hub Facility (SHF), Vehicle Maintenan...","[BEAUMONT TX LPC, VMF]","148,347",11/1/1987,5,"[56, 57, 58, 59, 62]",[77707-9998],"[Mail Processing, Network Facilities, Post Office, Vehicle Maintenance]","[Logistics (LOG), Main Post Office, Processing and Distribution Center/Facility (PDC/PDF), Service Hub Facility (SHF), Vehicle Maintenan...","[CONFLICTING_SQFT, MULTIPLE_UNIT_NAMES]",True,NaN
4,TEXAS 1,DALLAS,2400 DFW TURNPIKE,DALLAS,DALLAS,TX,75398-0001,75398,0001,Owned,[],[],[VMF],0,7/1/1975,1,[189],[75398-0001],[],[],[UNKNOWN_FACILITY_TYPE],False,NaN
5,TEXAS 1,DALLAS,401 TOM LANDRY HWY,DALLAS,DALLAS,TX,75260-9998,75260,9998,Owned,"[Administrative Office, Mail Processing, Network Facilities, Post Office, Vehicle Maintenance]","[Carrier Annex (ANX), Computerized Forwarding System (CFS), Finance Station - No Delivery, Logistics (LOG), Main Post Office, Office of ...","[P&DC, VMF]","439,959",7/1/1978,8,"[169, 170, 171, 172, 173, 174, 175, 188]","[75260-9731, 75260-9998]","[Administrative Office, Mail Processing, Network Facilities, Other Customer Service, Post Office, Vehicle Maintenance]","[Carrier Annex (ANX), Computerized Forwarding System (CFS), Finance Station - No Delivery, Logistics (LOG), Main Post Office, Office of ...","[CONFLICTING_SQFT, MISSING_CENTROID_ZIP, MULTIPLE_UNIT_NAMES, MULTIPLE_ZIP_SUFFIXES]",True,NaN
6,TEXAS 3,EL PASO,8401 BOEING DR,EL PASO,EL PASO,TX,79910-9716,79910,9716,Owned,"[Mail Processing, Network Facilities, Post Office]","[Logistics (LOG), Main Post Office, Processing and Distribution Center/Facility (PDC/PDF), Station]","[P&DC, VMF AT P&DC]","227,088",7/1/1997,5,"[215, 216, 217, 218, 223]","[79910-9716, 79910-9998]","[Mail Processing, Network Facilities, Post Office]","[Logistics (LOG), Main Post Office, Processing and Distribution Center/Facility (PDC/PDF), Station]","[CONFLICTING_SQFT, MISSING_CENTROID_ZIP, MULTIPLE_UNIT_NAMES, MULTIPLE_ZIP_SUFFIXES]",True,NaN
7,TEXAS 3,FLUVANNA,930 10TH ST,SCURRY,FLUVANNA,TX,79517-9998,79517,9998,Owned,[],[],[MAIN OFFICE],803,5/1/1966,1,[238],[79517-9998],[],[],[UNKNOWN_FACILITY_TYPE],False,

### missing_centroid_df.head()

,District,PO Name,Property Address,County,City,ST,ZIP Code,zip5,zip_suffix,Ownership,facility_types,facility_subtypes,unit_names_seen,Int Sq Ft,Bldg Occu Date,source_row_count,source_row_ids,original_zip_codes,original_facility_types,original_facility_subtypes,review_reasons,has_known_facility_type,has_centroid_zip
8,TEXAS 3,AMARILLO,2301 ROSS ST,POTTER,AMARILLO,TX,79120-9998,79120,9998,Owned,"[Mail Processing, Network Facilities, Post Office, Vehicle Maintenance]","[Logistics (LOG), Main Post Office, Processing and Distribution Center/Facility (PDC/PDF), Vehicle Maintenance Facility (VMF)]","[PDC, VMF]","220,270",3/1/1977,4,"[12, 13, 14, 15]","[79120-9341, 79120-9998]","[Mail Processing, Network Facilities, Post Office, Vehicle Maintenance]","[Logistics (LOG), Main Post Office, Processing and Distribution Center/Facility (PDC/PDF), Vehicle Maintenance Facility (VMF)]","[CONFLICTING_SQFT, MISSING_CENTROID_ZIP, MULTIPLE_UNIT_NAMES, MULTIPLE_ZIP_SUFFIXES]",True,False
10,TEXAS 3,AMARILLO,505 E 9TH AVE,POTTER,AMARILLO,TX,79105-9998,79105,9998,Owned,[Post Office],[Finance Station],[DOWNTOWN STATION],"8,624",2/1/1978,1,[9],[79105-9998],[Post Office],[Finance Station],[MISSING_CENTROID_ZIP],True,False
37,TEXAS 3,AUSTIN,8225 CROSS PARK DR,TRAVIS,AUSTIN,TX,78710-9716,78710,9716,Owned,"[Administrative Office, Mail Processing, Network Facilities, Post Office]","[Finance Station, Logistics (LOG), Main Post Office, Office of the Inspector General Office (OIG), Processing and Distribution Center/Fa...",[P&DC],"338,562",5/1/1988,5,"[42, 43, 44, 45, 46]",[78710-9716],"[Administrative Office, Mail Processing, Network Facilities, Post Office]","[Finance Station, Logistics (LOG), Main Post Office, Office of the Inspector General Office (OIG), Processing and Distribution Center/Fa...",[MISSING_CENTROID_ZIP],True,False
56,TEXAS 1,BLUEGROVE,1720 FM 172,CLAY,BLUEGROVE,TX,76352-9998,76352,9998,Owned,[Post Office],[Remotely Managed Post Office (RMPO)],[MPO/MODULAR BUILDING],736,12/1/1996,1,[71],[76352-9998],[Post Office],[Remotely Managed Post Office (RMPO)],[MISSING_CENTROID_ZIP],True,False
91,TEXAS 2,CHRIESMAN,136 PRAIRIE ST,BURLESON,CHRIESMAN,TX,77838-5000,77838,5000,Owned,[Post Office],[Remotely Managed Post Office (RMPO)],[MPO/MODULAR BLDG],732,10/1/1998,1,[110],[77838-5000],[Post Office],[Remotely Managed Post Office (RMPO)],[MISSING_CENTROID_ZIP],True,False
95,TEXAS 1,CLAYTON,4535 W HIGHWAY 315,PANOLA,CLAYTON,TX,75637-9998,75637,9998,Owned,[Post Office],[Remotely Managed Post Office (RMPO)],[MPO/MODULAR BLDG],736,5/1/1996,1,[114],[75637-9998],[Post Office],[Remotely Managed Post Office (RMPO)],[MISSING_CENTROID_ZIP],True,False
101,TEXAS 2,COLLEGE STATION,104 N HOUSTON ST,BRAZOS,COLLEGE STATION,TX,77841-9800,77841,9800,Owned,[Post Office],[Finance Station],[NORTHGATE STATION],"22,642",6/1/1937,1,[121],[77841-9800],[Post Office],[Finance Station],[MISSING_CENTROID_ZIP],True,False
116,TEXAS 2,CORPUS CHRISTI,809 NUECES BAY BLVD,NUECES,CORPUS CHRISTI,TX,78469-9998,78469,9998,Owned,"[Mail Processing, Network Facilities, Post Office, Vehicle Maintenance]","[Finance Station, Logistics (LOG), Main Post Office, Processing and Distribution Center/Facility (PDC/PDF), Vehicle Maintenance Facility...",[P&DC],"113,333",12/1/1969,5,"[133, 134, 135, 136, 137]",[78469-9998],"[Mail Processing, Network Facilities, Post Office, Vehicle Maintenance]","[Finance Station, Logistics (LOG), Main Post Office, Processing and Distribution Center/Facility (PDC/PDF), Vehicle Maintenance Facility...",[MISSING_CENTROID_ZIP],True,False
138,TEXAS 1,DALLAS,2400 DALLAS FT WORTH TPKE,DALLAS,DALLAS,TX,75398-9100,75398,9100,Owned,[Network Facilities],"[Logistics (LOG), Network Distribution Center (NDC/ASF)]",[NDC],"547,059",7/1/1975,2,"[164, 165]",[75398-9100],[Network Facilities],"[Logistics (LOG), Network Distribution Center (NDC/ASF)]",[MISSING_CENTROID_ZIP],True,False
146,TEXAS 1,DALLAS,401 TOM LANDRY HWY,DALLAS,DALLAS,TX,75260-9998,75260,9998,Owned,"[Administrative Office, Mail P

### final_df.head()

,District,PO Name,Property Address,County,City,ST,ZIP Code,zip5,zip_suffix,Ownership,facility_types,facility_subtypes,unit_names_seen,Int Sq Ft,Bldg Occu Date,source_row_count,source_row_ids,original_zip_codes,original_facility_types,original_facility_subtypes,review_reasons,has_known_facility_type,has_centroid_zip
0,TEXAS 3,ABILENE,2501 BUFFALO GAP RD,TAYLOR,ABILENE,TX,79605-6185,79605,6185,Owned,[Post Office],[Station],[SOUTHERN HILLS STA],"17,130",12/1/1984,1,[3],[79605-6185],[Post Office],[Station],[],True,True
1,TEXAS 3,ABILENE,341 PINE ST,TAYLOR,ABILENE,TX,79601-9998,79601,9998,Owned,[Post Office],"[Main Post Office, Station]",[MAIN OFFICE],"122,503",3/1/1936,2,"[1, 2]",[79601-9998],[Post Office],"[Main Post Office, Station]",[],True,True
2,TEXAS 3,ABILENE,810 N 4TH ST,TAYLOR,ABILENE,TX,79601-5924,79601,5924,Owned,[Vehicle Maintenance],[Vehicle Maintenance Facility (VMF)],[AUXILIARY VMF],"4,370",3/1/1987,1,[0],[79601-5924],[Vehicle Maintenance],[Vehicle Maintenance Facility (VMF)],[],True,True
3,TEXAS 1,ADDISON,4900 AIRPORT PKWY,DALLAS,ADDISON,TX,75001-9998,75001,9998,Owned,[Post Office],[Main Post Office],[MAIN OFFICE],"18,564",8/1/1998,1,[4],[75001-9998],[Post Office],[Main Post Office],[],True,True
4,TEXAS 2,ALAMO,423 LOS ALAMOS DR,HIDALGO,ALAMO,TX,78516-9998,78516,9998,Owned,[Post Office],[Main Post Office],[MAIN OFFICE],"5,049",NaN,1,[5],[78516-9998],[Post Office],[Main Post Office],[],True,True
5,TEXAS 2,ALICE,401 E 2ND ST,JIM WELLS,ALICE,TX,78332-9998,78332,9998,Owned,[Post Office],[Administrative Post Office (APO)],[MAIN OFFICE],"24,262",6/1/1966,1,[6],[78332-9998],[Post Office],[Administrative Post Office (APO)],[],True,True
6,TEXAS 1,ALLEN,401 CENTURY PKWY,COLLIN,ALLEN,TX,75013-8043,75013,8043,Owned,[Post Office],[Main Post Office],[MAIN OFFICE],"24,382",10/31/2020,1,[7],[75013-8043],[Post Office],[Main Post Office],[],True,True
7,TEXAS 2,ALVIN,455 E HOUSE ST,BRAZORIA,ALVIN,TX,77511-9998,77511,9998,Owned,[Post Office],[Main Post Office],[MAIN OFFICE],"17,058",9/1/1986,1,[8],[77511-9998],[Post Office],[Main Post Office],[],True,True
9,TEXAS 3,AMARILLO,5000 S WESTERN ST,POTTER,AMARILLO,TX,79109-6192,79109,6192,Owned,[Post Office],[Station],[LONE STAR STATION],"35,348",9/1/1999,1,[11],[79109-6192],[Post Office],[Station],[],True,True
11,TEXAS 3,AMARILLO,8301 W AMARILLO BLVD,POTTER,AMARILLO,TX,79124-2153,79124,2153,Owned,[Post Office],[Station],[JORDAN STATION],"26,220",5/1/2001,1,[10],[79124-2153],[Post Office],[Station],[],True,True
